# K3 · Canonical volumes (CPU)
Settings: **Internet OFF**, accelerator **None**.
Inputs: competition data **and the K1 output** (*Add Input*, *Your Work* tab, notebook K1).

For every training study: 5 slots, 64 slices, 140 mm crop, 224 x 224 pixels, per-series normalisation.
Size 224 (instead of 256) so that the 4,407 volumes fit within the 20 GB output limit: about 14 GB.

Outputs: `volumes_224.npy` (4407, 64, 224, 224) uint8, `masks.npy` (4407, 64), `index.csv` (study order, chosen series).

In [ ]:
import glob, os, sys, subprocess

def _walk_inputs(max_depth=5):
    """Walk /kaggle/input without ever entering the competition data (820,000 DICOM files)."""
    base = "/kaggle/input"
    for root, dirs, files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        dirs[:] = [d for d in dirs if d != "competitions" and "rsna-knee" not in d
                   and not d.endswith("_series") and depth < max_depth]
        yield root, dirs, files

def find(name):
    """Path of a file produced by another notebook attached as input (any mount depth)."""
    for root, dirs, files in _walk_inputs():
        if name in files:
            return os.path.join(root, name)
    p = f"/kaggle/working/{name}"
    return p if os.path.exists(p) else None

print("Attached inputs:")
for root, dirs, files in _walk_inputs(max_depth=3):
    if files:
        print("  ", root, "->", sorted(files)[:6])

wheels = [os.path.join(r, f) for r, d, fs in _walk_inputs() if os.path.basename(r) == "wheels" for f in fs if f.endswith(".whl")]
print(len(wheels), "wheels")
if wheels:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-index", "--no-deps"] + wheels, check=False)
common = find("knee_common.py")
if common is None:
    raise FileNotFoundError("knee_common.py not found: attach the K1 output (Add Input > Your Work > K1 series inventory).")
sys.path.insert(0, os.path.dirname(common))
print("shared code:", common)
import knee_common as kc

In [ ]:
import time, numpy as np, pandas as pd
from multiprocessing import Pool

COMP = kc.find_comp_dir()
SIZE = 224
inv = pd.read_csv(find("series_inventory.csv"))
slices = pd.read_parquet(find("slices.parquet")).sort_values(["SeriesInstanceUID", "rank"])
train = pd.read_csv(f"{COMP}/train.csv")

SORTED = {}
ps = inv.set_index("SeriesInstanceUID")[["ps_r", "ps_c"]]
for sid, g in slices.groupby("SeriesInstanceUID", sort=False):
    SORTED[sid] = (g["file"].tolist(), float(ps.at[sid, "ps_r"]), float(ps.at[sid, "ps_c"]))
INV_BY_STUDY = {st: g for st, g in inv.groupby("StudyInstanceUID")}
studies = train["StudyInstanceUID"].tolist()
print(len(studies), "studies,", len(SORTED), "sorted series")

In [ ]:
def one(i):
    st = studies[i]
    g = INV_BY_STUDY.get(st)
    if g is None:
        return i, np.zeros((kc.N_SLICES, SIZE, SIZE), np.uint8), np.zeros(kc.N_SLICES, bool), [None] * 5
    sub = {sid: SORTED[sid] for sid in g["SeriesInstanceUID"] if sid in SORTED}
    vol, mask, chosen = kc.build_volume(COMP, "train", st, g, size=SIZE, sorted_files=sub)
    return i, vol, mask, chosen

# Quick test before the full run
t0 = time.time(); _, v, m, c = one(0); print(v.shape, m.sum(), c, f"{time.time()-t0:.1f} s per study")

In [ ]:
N = len(studies)
vols = np.lib.format.open_memmap("/kaggle/working/volumes_224.npy", mode="w+", dtype=np.uint8,
                                 shape=(N, kc.N_SLICES, SIZE, SIZE))
masks = np.zeros((N, kc.N_SLICES), dtype=bool)
chosen_all = [None] * N
t0 = time.time()
with Pool(os.cpu_count()) as pool:
    for k, (i, v, m, c) in enumerate(pool.imap_unordered(one, range(N), chunksize=4)):
        vols[i] = v; masks[i] = m; chosen_all[i] = c
        if (k + 1) % 250 == 0:
            el = time.time() - t0
            print(f"{k+1}/{N}  {el/60:.1f} min  about {el/(k+1)*(N-k-1)/60:.0f} min left")
vols.flush(); del vols
np.save("/kaggle/working/masks.npy", masks)
idx = pd.DataFrame({"StudyInstanceUID": studies, "row": np.arange(N), "n_valid": masks.sum(1)})
for j, (name, *_) in enumerate(kc.SLOTS):
    idx[name] = [c[j] if c else None for c in chosen_all]
idx.to_csv("/kaggle/working/index.csv", index=False)
print("done in", f"{(time.time()-t0)/60:.0f} min")
print(idx["n_valid"].describe())

In [ ]:
# Visual check of 4 volumes
import matplotlib.pyplot as plt
V = np.load("/kaggle/working/volumes_224.npy", mmap_mode="r")
fig, ax = plt.subplots(4, 8, figsize=(16, 8))
for r, i in enumerate(np.random.RandomState(0).choice(N, 4, replace=False)):
    for c, s in enumerate(range(0, 64, 8)):
        ax[r, c].imshow(V[i, s], cmap="gray"); ax[r, c].axis("off")
plt.tight_layout(); plt.savefig("/kaggle/working/check_volumes.png", dpi=60); plt.show()